# Model to transform the gas to an hydrogen grid

Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

Import Data

In [6]:
# Specify the path to your Excel file
input_file_path = '../01_data/01_input_data/02_processed/'
excel_file_path = 'Data.xlsx'  

# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(input_file_path + excel_file_path, sheet_name='Nodes')
df_commodities = pd.read_excel(input_file_path + excel_file_path, sheet_name='Commodities')
df_edges = pd.read_excel(input_file_path + excel_file_path, sheet_name='Edges')
df_parameter = pd.read_excel(input_file_path + excel_file_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Supply')
df_demand_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Demand')

Create input data structure

In [7]:
# Extract nodes, edges and commodities from the DataFrames
Supply_nodes = df_nodes['Supply Nodes'].dropna().tolist()
Demand_nodes = df_nodes['Demand Nodes'].dropna().tolist()
Commodities = df_commodities['Commodities'].dropna().tolist()
Edges = list(zip(df_edges['Source'], df_edges['Destination']))

# Create a nested dictionary for initial capacities
Initial_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    initial_capacity = row['initial_capacities']

    edge = f"{source}{destination}"

    if commodity not in Initial_capacities:
        Initial_capacities[commodity] = {}

    Initial_capacities[commodity][edge] = initial_capacity

# Create a nested dictionary for max capacities
Max_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    max_capacity = row['max_capacities']

    edge = f"{source}{destination}"

    if commodity not in Max_capacities:
        Max_capacities[commodity] = {}

    Max_capacities[commodity][edge] = max_capacity

# Create a nested dictionary for edge cost
Edge_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    edge_cost = row['costs_edge']

    edge = f"{source}{destination}"

    if commodity not in Edge_cost:
        Edge_cost[commodity] = {}

    Edge_cost[commodity][edge] = edge_cost

# Create a nested dictionary for new pipelines
Pipe_new_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    new_cost = row['new_build_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_new_cost:
        Pipe_new_cost[commodity] = {}

    Pipe_new_cost[commodity][edge] = new_cost

# Create a nested dictionary for pipeline conversion
Pipe_conv_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cost = row['conversion_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_cost:
        Pipe_conv_cost[commodity] = {}

    Pipe_conv_cost[commodity][edge] = conv_cost

# Create a nested dictionary for supply values, skipping 0 and NaN values
Supply_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    supply_node = row['Node']
    supply_value = row['Supply']

    if commodity not in Supply_values:
        Supply_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(supply_value) and supply_value != 0:
        Supply_values[commodity][supply_node] = supply_value

# Create a nested dictionary for demand values, skipping 0 and NaN values
Demand_values = {}
for index, row in df_demand_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    demand_value = row['Demand']

    if commodity not in Demand_values:
        Demand_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(demand_value) and demand_value != 0:
        Demand_values[commodity][demand_node] = demand_value

Print data structure for control

In [8]:
# Print the data
print("Supply Nodes:", Supply_nodes)
print("Demand Nodes:", Demand_nodes)
print("Commodities:", Commodities)
print("Edges:", Edges)
print("Initial Capacities:", Initial_capacities)
print("Max Capacities:", Max_capacities)
print("Costs per edge Capacities:", Edge_cost)
print("Costs for new pipelines:", Pipe_new_cost)
print("Costs for convert pipelines:", Pipe_conv_cost)
print("Supply Values:", Supply_values)
print("Demand Values:", Demand_values)

Supply Nodes: ['S1', 'S2', 'S3']
Demand Nodes: ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10']
Commodities: ['Methane', 'Hydrogen']
Edges: [('S1', 'D1'), ('S1', 'D2'), ('S2', 'D2'), ('S2', 'D3'), ('S3', 'D1'), ('D1', 'D2'), ('D3', 'D4'), ('D4', 'D5'), ('D4', 'D6'), ('D6', 'D7'), ('D5', 'D8'), ('D5', 'D9'), ('D9', 'D10'), ('D7', 'D10')]
Initial Capacities: {'Methane': {'S1D1': 20, 'S1D2': 30, 'S2D2': 25, 'S2D3': 40, 'S3D1': 35, 'D1D2': 30, 'D3D4': 30, 'D4D5': 40, 'D4D6': 40, 'D6D7': 30, 'D5D8': 15, 'D5D9': 10, 'D9D10': 10, 'D7D10': 20}, 'Hydrogen': {'S1D1': 0, 'S1D2': 0, 'S2D2': 0, 'S2D3': 0, 'S3D1': 0, 'D1D2': 0, 'D3D4': 0, 'D4D5': 0, 'D4D6': 0, 'D6D7': 0, 'D5D8': 0, 'D5D9': 0, 'D9D10': 0, 'D7D10': 0}}
Max Capacities: {'Methane': {'S1D1': 25, 'S1D2': 40, 'S2D2': 35, 'S2D3': 50, 'S3D1': 45, 'D1D2': 30, 'D3D4': 30, 'D4D5': 40, 'D4D6': 40, 'D6D7': 30, 'D5D8': 15, 'D5D9': 10, 'D9D10': 10, 'D7D10': 20}, 'Hydrogen': {'S1D1': 25, 'S1D2': 40, 'S2D2': 35, 'S2D3': 50, 'S3D1': 45, '

Create model

In [9]:
# Create a new model
model = gp.Model("Grid_Transformation")

Set parameter Username
Academic license - for non-commercial use only - expires 2024-12-20


Define parameters

In [10]:
# Parameters
supply_nodes = Supply_nodes  # Supply nodes
demand_nodes = Demand_nodes  # Demand nodes
commodities = Commodities  # Commodity types
edges = Edges  # Edges
initial_capacities = Initial_capacities # Initial capacities
max_capacities = Max_capacities  # Maximum capacities
costs_edge = Edge_cost  # Cost to transport from node to node
capacity_new_cost = Pipe_new_cost  # Cost to increase capacity
capacity_change_cost = Pipe_conv_cost  # Cost to increase capacity

supply_values = Supply_values  # Supply values
demand_values = Demand_values  # Demand values

Define decision variables

In [11]:
# Decision variables
x = {}
y = {}
z = {}
Change = {}
for commodity in commodities:
    x[commodity] = {}
    y[commodity] = {}
    z[commodity] = {}
    Change[commodity] = model.addVar(vtype=GRB.BINARY, name=f"w_{commodity}")  # Binary variable for switching
    for edge in edges:
        x[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_{commodity}_{edge[0]}_{edge[1]}")
        y[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_{commodity}_{edge[0]}_{edge[1]}")
        z[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_{commodity}_{edge[0]}_{edge[1]}")

Define objective and constraints

In [12]:
# Objective function (minimize total transportation cost + cost to increase capacity)
model.setObjective(
    gp.quicksum(x[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges) +
    gp.quicksum(y[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges) +
    gp.quicksum(z[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges),
    GRB.MINIMIZE
)

# Constraints

# Supply constraints
for node in supply_nodes:
    for commodity in commodities:
        model.addConstr(gp.quicksum(x[commodity][edge] for edge in edges if edge[0] == node) 
                        <= supply_values[commodity][node], f"supply_{commodity}_{node}")

# Demand constraints
for node in demand_nodes:
    for commodity in commodities:
        model.addConstr(gp.quicksum(x[commodity][edge] for edge in edges if edge[1] == node) 
                        == demand_values[commodity][node], f"demand_{commodity}_{node}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in edges:
        model.addConstr(x[commodity][edge] 
                        <= y[commodity][edge] + z[commodity][edge]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")

# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in edges:
        model.addConstr(y[commodity][edge] + z[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"total_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraint for switch
model.addConstr(Change[Commodities[0]] + Change[Commodities[1]] == 1, "switching_constraint")
for commodity in commodities:
    for edge in edges:
        model.addConstr(initial_capacities[Commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity] 
                        == z[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in edges:
        model.addConstr(x[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

Optimize the model

In [13]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 139 rows, 86 columns and 264 nonzeros
Model fingerprint: 0x00292fe8
Variable types: 84 continuous, 2 integer (2 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [5e-01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Presolve removed 123 rows and 61 columns
Presolve time: 0.00s
Presolved: 16 rows, 25 columns, 48 nonzeros
Variable types: 24 continuous, 1 integer (1 binary)
Found heuristic solution: objective 1939.5000000

Root relaxation: cutoff, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff

Results processing

In [14]:
# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for commodity in commodities:
        for edge in edges:
            print(f"{commodity}, {edge}: Flow of Commodity = {x[commodity][edge].x}, New Capacity = {y[commodity][edge].x}, Switched = {Change[commodity].x}, Changed Capacity = {z[commodity][edge].x}")
    print("****************************")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")

Optimal solution found!
Methane, ('S1', 'D1'): Flow of Commodity = 15.0, New Capacity = 15.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('S1', 'D2'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('S2', 'D2'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('S2', 'D3'): Flow of Commodity = 25.0, New Capacity = 25.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('S3', 'D1'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('D1', 'D2'): Flow of Commodity = 20.0, New Capacity = 20.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('D3', 'D4'): Flow of Commodity = 1.0, New Capacity = 1.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('D4', 'D5'): Flow of Commodity = 20.0, New Capacity = 20.0, Switched = -0.0, Changed Capacity = -0.0
Methane, ('D4', 'D6'): Flow of Commodity = 5.0, New Capacity = 5.0, Switched = -0.0, Changed Cap